# 02 - Preprocessing and the leakage-safe split

From the EDA I have three things to act on: drop the rows that have no `profile_id` (they can't be grouped), handle the `u_d`/`motor_speed` gaps with imputation that is fit on training data only, and split so that whole sessions are held out rather than individual rows. This notebook walks through that and verifies the split doesn't leak.

In [1]:
import sys
sys.path.append("..")

import numpy as np
from src.data_prep import load_raw, clean, split_by_profile
from src.features import make_preprocessor
from src.config import GROUP_COL

raw = load_raw()
raw.shape

(1048575, 9)

## Drop ungroupable rows

In [2]:
df = clean(raw)
print("dropped rows:", len(raw) - len(df))
print("remaining rows:", len(df))

dropped rows: 24089
remaining rows: 1024486


About 24k rows go, the ones missing `profile_id`. What's left still contains the `u_d`/`motor_speed` gaps, which I leave in place on purpose: the imputer in the model pipeline will fill them. That keeps prediction able to handle a record with a missing field instead of refusing to score it.

## Split by session

In [3]:
X_train, X_test, y_train, y_test, groups_train = split_by_profile(df)
print("train rows:", len(X_train))
print("test rows :", len(X_test))
print("test share:", round(len(X_test) / len(df), 3))

train rows: 836542
test rows : 187944
test share: 0.183


## Leakage check: no session in both train and test

In [4]:
train_sessions = set(df.iloc[X_train.index][GROUP_COL])
test_sessions = set(df.iloc[X_test.index][GROUP_COL])
overlap = train_sessions & test_sessions

print("train sessions:", len(train_sessions))
print("test sessions :", len(test_sessions))
print("overlapping    :", len(overlap))
assert len(overlap) == 0, "sessions leak across the split"
print("OK - no session appears on both sides")

train sessions: 43
test sessions : 11
overlapping    : 0
OK - no session appears on both sides


Whole sessions go to one side or the other, so the test rows come from motor runs the model never saw in training. That is the condition for the test score to reflect real generalisation. The row share lands near the requested 20% even though we split on sessions, because the sessions are of similar order of magnitude in size.

## Train and test targets are comparable

In [5]:
import pandas as pd
pd.DataFrame({"train": y_train.describe(), "test": y_test.describe()}).round(2)

,train,test
count,836542.00,187944.00
mean,55.98,61.07
std,20.13,18.90
min,21.03,20.86
25%,38.33,45.18
50%,55.61,64.15
75%,70.56,75.05
max,113.61,96.78


The two `pm` distributions are close enough that the holdout is a fair test of the same problem, not a harder or easier one. Some difference is expected and healthy, since we're holding out entire runs rather than reshuffling rows.

## Preprocessor, fit on train only

In [6]:
pre = make_preprocessor(scale=True)
Xt = pre.fit_transform(X_train)
print("any NaN left after imputation:", bool(np.isnan(Xt).any()))
print("feature means after scaling (train):", np.round(Xt.mean(axis=0), 3))
print("feature stds after scaling (train) :", np.round(Xt.std(axis=0), 3))

any NaN left after imputation: False
feature means after scaling (train): [ 0. -0.  0.  0.  0. -0.  0.]
feature stds after scaling (train) : [1. 1. 1. 1. 1. 1. 1.]


After fitting on the training set the imputer has filled every gap and the scaler has centred and unit-scaled each feature. In the model pipelines this same fitted preprocessing is applied to the test set, the imputation values and scaling statistics come from train alone, so the test set stays untouched during fitting. The split function, this preprocessing, and the feature list are all imported from `src`, which is exactly what `train.py` and `predict.py` will use, so the two stay in step.